# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

# 2. File Paths

In [ ]:
# Path ke file data mentah MovieLens 100K
raw_data_path = '../Data/ml-100k/u.data'

# Path folder output untuk menyimpan hasil preprocessing
output_dir = '../Data/TrainTest'

print(f"[INFO] Raw data path : {raw_data_path}")
print(f"[INFO] Output dir    : {output_dir}")

# 3. Load Data

In [ ]:
# Kolom pada file u.data: user_id, item_id, rating, timestamp
# File menggunakan tab sebagai separator
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(raw_data_path, sep='\t', names=column_names)

print(f"[INFO] Total baris data : {len(df):,}")
print(f"[INFO] Kolom            : {list(df.columns)}")
df.head()

# 4. Preprocess Data

## 1. Buat Matriks Interaksi User-Item

In [ ]:
# Periksa jumlah pengguna dan film unik
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()

print(f"[INFO] Jumlah pengguna unik : {n_users}")
print(f"[INFO] Jumlah film unik     : {n_items}")
print(f"[INFO] Total rating         : {len(df):,}")

In [ ]:
# Pivot tabel menjadi matriks user x item
# Sel yang tidak memiliki rating diisi dengan 0
interaction_matrix = df.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)

print(f"[INFO] Dimensi matriks interaksi: {interaction_matrix.shape}")
interaction_matrix.head()

## 2. Normalisasi Rating

In [ ]:
# Normalisasi rating ke rentang [0, 1] dengan membagi nilai maksimum rating (5.0)
# Sel kosong (nilai 0) tetap bernilai 0 setelah normalisasi,
# sehingga dapat dibedakan dari rating terendah (1/5 = 0.2)
MAX_RATING = 5.0

normalized_matrix = interaction_matrix.values / MAX_RATING

print(f"[INFO] Nilai minimum (non-zero) : {normalized_matrix[normalized_matrix > 0].min():.2f}")
print(f"[INFO] Nilai maksimum           : {normalized_matrix.max():.2f}")
print(f"[INFO] Dimensi                  : {normalized_matrix.shape}")

## 3. Train / Validation / Test Split (Hold-Out)

Split dilakukan pada level rating individual, bukan pada level pengguna.
Hal ini memastikan setiap pengguna dapat muncul di ketiga partisi.

Proporsi split:
- **Train (70%)** — digunakan untuk melatih VAE dan RSVD
- **Validation (10%)** — digunakan untuk mencari bobot ensemble optimal (alpha)
  tanpa melihat test set, sehingga evaluasi akhir tetap tidak bias
- **Test (20%)** — hanya disentuh satu kali untuk pelaporan performa akhir

In [ ]:
# Konfigurasi split
VAL_RATIO  = 0.10  # 10% untuk validasi
TEST_RATIO = 0.20  # 20% untuk test
RANDOM_SEED = 42

# Ambil koordinat semua rating yang ada nilainya
all_rows, all_cols = np.nonzero(normalized_matrix)
total_ratings = len(all_rows)

n_test = int(total_ratings * TEST_RATIO)
n_val  = int(total_ratings * VAL_RATIO)
n_train = total_ratings - n_test - n_val

print(f"[INFO] Total rating   : {total_ratings:,}")
print(f"[INFO] Train          : {n_train:,} ({n_train/total_ratings*100:.0f}%)")
print(f"[INFO] Validation     : {n_val:,}  ({n_val/total_ratings*100:.0f}%)")
print(f"[INFO] Test           : {n_test:,} ({n_test/total_ratings*100:.0f}%)")

In [ ]:
# Acak seluruh indeks rating dengan seed tetap untuk reproduksibilitas
np.random.seed(RANDOM_SEED)
shuffled_idx = np.random.permutation(total_ratings)

# Potong indeks menjadi tiga bagian berdasarkan proporsi
test_idx  = shuffled_idx[:n_test]
val_idx   = shuffled_idx[n_test:n_test + n_val]
train_idx = shuffled_idx[n_test + n_val:]

# Ambil koordinat (baris, kolom) masing-masing partisi
test_rows,  test_cols  = all_rows[test_idx],  all_cols[test_idx]
val_rows,   val_cols   = all_rows[val_idx],   all_cols[val_idx]
train_rows, train_cols = all_rows[train_idx], all_cols[train_idx]

print(f"[INFO] Koordinat test diekstrak  : {len(test_rows):,} rating")
print(f"[INFO] Koordinat val diekstrak   : {len(val_rows):,} rating")
print(f"[INFO] Koordinat train diekstrak : {len(train_rows):,} rating")

In [ ]:
# Bangun matriks train: mulai dari matriks penuh, hapus rating val dan test
train_data = normalized_matrix.copy().astype(np.float32)
train_data[val_rows,  val_cols]  = 0.0
train_data[test_rows, test_cols] = 0.0

# Bangun matriks validation: mulai dari nol, isi hanya rating val
val_data = np.zeros(normalized_matrix.shape, dtype=np.float32)
val_data[val_rows, val_cols] = normalized_matrix[val_rows, val_cols]

# Bangun matriks test: mulai dari nol, isi hanya rating test
test_data = np.zeros(normalized_matrix.shape, dtype=np.float32)
test_data[test_rows, test_cols] = normalized_matrix[test_rows, test_cols]

# Verifikasi: jumlah rating di masing-masing matriks harus sesuai
n_train_check = int(np.sum(train_data > 0))
n_val_check   = int(np.sum(val_data   > 0))
n_test_check  = int(np.sum(test_data  > 0))

print(f"[VERIFIKASI] Train rating : {n_train_check:,}")
print(f"[VERIFIKASI] Val rating   : {n_val_check:,}")
print(f"[VERIFIKASI] Test rating  : {n_test_check:,}")
print(f"[VERIFIKASI] Total        : {n_train_check + n_val_check + n_test_check:,} (harus {total_ratings:,})")

# 5. Save Data

In [ ]:
import os

# Buat folder output jika belum ada
os.makedirs(output_dir, exist_ok=True)

# Simpan matriks train (70%): dipakai untuk melatih VAE dan RSVD di Training.ipynb
np.save(os.path.join(output_dir, 'train_data'), train_data)

# Simpan matriks validation (10%): dipakai untuk mencari alpha ensemble di Testing.ipynb
np.save(os.path.join(output_dir, 'val_data'), val_data)

# Simpan matriks test (20%): hanya disentuh satu kali untuk pelaporan performa akhir
np.save(os.path.join(output_dir, 'test_data'), test_data)

# Simpan matriks rating penuh sebelum di-split
# Dipakai oleh CrossValidation.ipynb agar setiap fold bisa membuat split-nya sendiri
np.save(os.path.join(output_dir, 'normalized_matrix'), normalized_matrix)

print(f"[SUCCESS] train_data.npy       disimpan ({n_train_check:,} rating)")
print(f"[SUCCESS] val_data.npy         disimpan ({n_val_check:,} rating)")
print(f"[SUCCESS] test_data.npy        disimpan ({n_test_check:,} rating)")
print(f"[SUCCESS] normalized_matrix.npy disimpan ({total_ratings:,} rating)")